In [1]:
from ta.volatility import AverageTrueRange
from ta.momentum import RSIIndicator
from ta.trend import EMAIndicator, SMAIndicator

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from backtesting import Strategy
from backtesting import Backtest


from datetime import timedelta

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [2]:
# swing structure
def implement_market_structure_states_strict(df, n=5):
    """Calcule les états du marché avec transition obligatoire par l'état NEUTRAL.

    Aucun passage direct entre BULLISH et BEARISH n'est autorisé.
    """
    df = df.copy()
    length = len(df)

    df['is_swing_high'] = False
    df['is_swing_low'] = False
    for k in range(n, len(df) - n):
        # Le plus haut des n bougies précédentes et suivantes
        high_range = df['High'].iloc[k - n:k + n + 1]
        if df['High'].iloc[k] == high_range.max():
            df.loc[df.index[k], 'is_swing_high'] = True
            
        # Le plus bas des n bougies précédentes et suivantes
        low_range = df['Low'].iloc[k - n:k + n + 1]
        if df['Low'].iloc[k] == low_range.min():
            df.loc[df.index[k], 'is_swing_low'] = True

    states = []
    current_state = 'NEUTRAL'

    # Variables de structure de marché
    last_confirmed_high = None
    last_confirmed_low = None
    higher_low = None  # Le plancher (uniquement actif en BULLISH)
    lower_high = None  # Le plafond (uniquement actif en BEARISH)

    for j in range(length):
        close_p = df["Close"].iloc[j]

        # --- ÉTAPE B : Machine à états avec transit strict par NEUTRAL ---
        if current_state == "BULLISH":
            # Sortie de Bullish : Clôture sous le Higher Low (HL) -> NEUTRAL uniquement
            if higher_low is not None and close_p < higher_low:
                current_state = "NEUTRAL"
                higher_low = None  # Reset des repères obsolètes
                lower_high = None

        elif current_state == "BEARISH":
            # Sortie de Bearish : Clôture au-dessus du Lower High (LH) -> NEUTRAL uniquement
            if lower_high is not None and close_p > lower_high:
                current_state = "NEUTRAL"
                higher_low = None
                lower_high = None  # Reset des repères obsolètes

        elif current_state == "NEUTRAL":
            # En zone neutre, on attend un NOUVEAU BOS pour définir la tendance
            # Entrée en Bullish : Clôture au-dessus du dernier sommet confirmé
            # il faut tjs un high and un low avant de former une structure bearish ou bullish
            if last_confirmed_high is not None and last_confirmed_low is not None and close_p > last_confirmed_high:
                current_state = "BULLISH"
                higher_low = (
                    last_confirmed_low  # On fige le dernier creux comme HL
                )

            # Entrée en Bearish : Clôture sous le dernier creux confirmé
            elif last_confirmed_high is not None and last_confirmed_low is not None and close_p < last_confirmed_low:
                current_state = "BEARISH"
                lower_high = (
                    last_confirmed_high  # On fige le dernier sommet comme LH
                )

        states.append(current_state)

         # --- ÉTAPE A : Enregistrement des Swings dès qu'ils se confirment ---
        if df["is_swing_high"].iloc[j] == True:
            last_confirmed_high = df["High"].iloc[j]
            if current_state == "BEARISH" and lower_high > last_confirmed_high:
                lower_high = last_confirmed_high

        if df["is_swing_low"].iloc[j] == True:
            last_confirmed_low = df["Low"].iloc[j]
            if current_state == "BULLISH" and higher_low < last_confirmed_low:
                higher_low = last_confirmed_low

    df["market_state"] = states
    return df


In [3]:
# swing structure direct Bullish to Bearish
def implement_market_structure_states_direct(df, n=5):
    """Calcule les états du marché avec transition obligatoire par l'état NEUTRAL.

    Aucun passage direct entre BULLISH et BEARISH n'est autorisé.
    """
    df = df.copy()
    length = len(df)

    df['is_swing_high'] = False
    df['is_swing_low'] = False
    for k in range(n, len(df) - n):
        # Le plus haut des n bougies précédentes et suivantes
        high_range = df['High'].iloc[k - n:k + n + 1]
        if df['High'].iloc[k] == high_range.max():
            df.loc[df.index[k], 'is_swing_high'] = True
            
        # Le plus bas des n bougies précédentes et suivantes
        low_range = df['Low'].iloc[k - n:k + n + 1]
        if df['Low'].iloc[k] == low_range.min():
            df.loc[df.index[k], 'is_swing_low'] = True

    states = []
    current_state = None

    # Variables de structure de marché
    last_confirmed_high = None
    last_confirmed_low = None
    higher_low = None  # Le plancher (uniquement actif en BULLISH)
    lower_high = None  # Le plafond (uniquement actif en BEARISH)

    for j in range(length):
        close_p = df["Close"].iloc[j]

        # --- ÉTAPE B : Machine à états avec transit strict par NEUTRAL ---
        if current_state == "BULLISH" or current_state is None:
            # Sortie de Bullish : Clôture sous le Higher Low (HL) -> NEUTRAL uniquement
            if higher_low is not None and close_p < higher_low:
                current_state = "BEARISH"

        elif current_state == "BEARISH" or current_state is None:
            # Sortie de Bearish : Clôture au-dessus du Lower High (LH) -> NEUTRAL uniquement
            if lower_high is not None and close_p > lower_high:
                current_state = "BULLISH"

        states.append(current_state)

         # --- ÉTAPE A : Enregistrement des Swings dès qu'ils se confirment ---
        if df["is_swing_high"].iloc[j] == True:
            last_confirmed_high = df["High"].iloc[j]
            lower_high = last_confirmed_high

        if df["is_swing_low"].iloc[j] == True:
            last_confirmed_low = df["Low"].iloc[j]
            higher_low = last_confirmed_low


    df["market_state"] = states
    return df


In [4]:
# define support level
def define_support_level(df, define_key_level_window = 20):
    # gap
    gap = 0.3

    # all resistant highs
    support_list = []
    
    for row in df.iterrows():
        start_index = row[0] - timedelta(days=define_key_level_window)
        back_candle_range = df.loc[start_index:row[0]]
        pivot_highs = back_candle_range[back_candle_range['is_swing_high'] == True]
        # print(pivot_highs)

        for candle in pivot_highs.iterrows():
            frequency = 0
            total_candle = 0
            # resistant highs related to current high
            support_zone = []

            for candle2 in pivot_highs.iterrows():
                # candle 2 date needs to be before candle date
                if (candle2[0] <= candle[0] and 
                    candle[1]['High'] + gap > candle2[1]['High'] and 
                    candle[1]['High'] - gap < candle2[1]['High']):
                    frequency += 1
                    total_candle += candle2[1]['High']
                    # support_zone.append(candle2)

            if frequency > 2:
                mean_candle = round(total_candle / frequency, 3)
                support_zone.append(candle)

                # for all new resistant, check if the candle is in resistant list, if not add it
                for i in range(len(support_zone)):
                    is_support = False

                    for j in range(len(support_list)):
                        if support_zone[i][0] == support_list[j][0][0]:
                            # print(f"{support_zone[i][0]} = {support_list[j][0][0]}")
                            is_support = True

                    if not is_support:
                        # print(f"{support_zone[i][0]} mean_candle = {mean_candle} & frequency = {frequency}")
                        resistant_candle = [support_zone[i], mean_candle]
                        support_list.append(resistant_candle)

    # print(resistant_list)
    return support_list


In [5]:
# define resistant level 
def define_resistant_level(df, days = 20):
    # gap
    gap = 0.3

    # all resistant highs
    resistant_list = []
    
    for row in df.iterrows():
        start_index = row[0] - timedelta(days=days)
        back_candle_range = df.loc[start_index:row[0]]
        pivot_lows = back_candle_range[back_candle_range['is_swing_low'] == True]
        # print(pivot_highs)

        for candle in pivot_lows.iterrows():
            frequency = 0
            total_candle = 0
            # resistant highs related to current high
            resistant_zone = []

            for candle2 in pivot_lows.iterrows():
                # candle 2 date needs to be before candle date
                if (candle2[0] <= candle[0] and 
                    candle[1]['Low'] + gap > candle2[1]['Low'] and 
                    candle[1]['Low'] - gap < candle2[1]['Low']):
                    frequency += 1
                    total_candle += candle2[1]['Low']
                    # resistant_zone.append(candle2)

            if frequency > 2:
                mean_candle = round(total_candle / frequency, 3)
                resistant_zone.append(candle)

                # for all new resistant, check if the candle is in resistant list, if not add it
                for i in range(len(resistant_zone)):
                    is_resistant = False

                    for j in range(len(resistant_list)):
                        if resistant_zone[i][0] == resistant_list[j][0][0]:
                            # print(f"{resistant_zone[i][0]} = {resistant_list[j][0][0]}")
                            is_resistant = True

                    if not is_resistant:
                        # print(f"{resistant_zone[i][0]} mean_candle = {mean_candle} & frequency = {frequency}")
                        resistant_candle = [resistant_zone[i], mean_candle]
                        resistant_list.append(resistant_candle)

    # print(resistant_list)
    return resistant_list


In [6]:
structure_d1_window = 2
structure_m15_window = 2

date_format = '%Y-%m-%d %H:%M'
# start_date = '2023-1-1 00:00'
# end_date = '2024-3-29 00:00'
start_date = datetime.strptime('2026-2-11 00:00', date_format)
end_date = datetime.strptime('2026-3-9 00:00', date_format)
# start_date = '2022-01-1 00:00'
# end_date = '2026-9-16 00:00'
pair = 'GBPJPY'
timezone = "Europe/London"

# nombre de jours pour les high.low
define_key_level_window = 15

# window for look back for key level
back_candles_window = 20

# gap t avoid look ahead
gap_look_ahead = 1

# nombre de candle au dessus de key level après breakou
breakout_candles_above = 9

# nombre de jours à chercher key level
key_level_lookback_days = 25

# vérifier si 30 candle dans ke passé pour valider un valid breakout,
# un valid breakout doit avoir {breakout_candles_above} candles au dessus du support level
breakout_candles_above_lookback = 30


In [7]:
df_d1_raw = pd.read_csv(f"../Trading Data/{pair}_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h4_raw = pd.read_csv(f"../Trading Data/{pair}_H4.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_m15_raw = pd.read_csv(f"../Trading Data/{pair}_M15.csv",sep=',', parse_dates=['Date'], index_col='Date')

print(df_d1_raw)
print(df_h4_raw)
print(df_m15_raw)

               Open     High      Low    Close  Tickvol
Date                                                   
2010-09-10  129.605  130.065  128.995  129.255   172229
2010-09-13  129.565  130.115  128.780  129.090   215859
2010-09-14  129.105  129.430  127.640  129.015   226170
2010-09-15  129.015  134.070  128.555  133.975   252317
2010-09-16  133.975  134.260  132.970  134.150   222443
...             ...      ...      ...      ...      ...
2026-09-18  208.323  211.276  208.233  210.095   364707
2026-09-21  210.296  210.762  209.676  210.321   243731
2026-09-22  210.321  210.892  209.513  210.080   333200
2026-09-23  210.080  210.176  209.422  209.564   273936
2026-09-24  209.566  210.105  208.768  209.848   199988

[4183 rows x 5 columns]
                        Open     High      Low    Close  Tickvol
Date                                                            
2010-09-10 04:00:00  129.605  129.630  128.995  129.385    25539
2010-09-10 08:00:00  129.380  129.985  129.275  129.

In [8]:
# process D1 data
df_d1 = df_d1_raw.loc[start_date - timedelta(days=30) : end_date]

df_d1 = implement_market_structure_states_strict(df_d1, n=structure_d1_window)
df_d1["state_changed"] = df_d1["market_state"] != df_d1[
    "market_state"
].shift(1)
print(df_d1[df_d1["state_changed"]][["Close", "market_state"]])
df_d1 = df_d1.loc[start_date : end_date]


              Close market_state
Date                            
2026-01-12  212.771      NEUTRAL
2026-02-11  208.614      BEARISH
2026-02-24  210.224      NEUTRAL
2026-02-25  211.860      BULLISH


In [9]:
# process h4 data
df_h4 = df_h4_raw.loc[start_date - timedelta(days=20) : end_date]
df_h4 = implement_market_structure_states_strict(df_h4, 5)
df_h4 = df_h4.loc[start_date : end_date]

h4_d1_direction_list = []
for i in range(len(df_d1)):
    row = df_d1.iloc[i]
    start_date_daily = row.name
    end_date_daily = start_date_daily + timedelta(hours=23, minutes=59)
    df_by_day = df_h4.loc[f'{start_date_daily}' : f'{end_date_daily}']

    for index2, row2 in df_by_day.iterrows():
        h4_d1_direction_list.append(df_d1.iloc[i-1]['market_state'])

df_h4['market_state_d1'] = h4_d1_direction_list

df_h4

,Open,High,Low,Close,Tickvol,is_swing_high,is_swing_low,market_state,market_state_d1
Date,,,,,,,,,
2026-02-11 00:00:00,210.561,210.662,209.800,209.801,34589,False,False,BEARISH,BULLISH
2026-02-11 04:00:00,209.800,209.971,208.980,209.346,27558,False,False,BEARISH,BULLISH
2026-02-11 08:00:00,209.347,210.179,209.267,210.129,32753,False,False,BEARISH,BULLISH
2026-02-11 12:00:00,210.130,210.519,208.642,209.540,67853,False,False,BEARISH,BULLISH
2026-02-11 16:00:00,209.540,209.608,208.175,208.280,56222,False,False,BEARISH,BULLISH
...,...,...,...,...,...,...,...,...,...
2026-03-06 08:00:00,210.848,210.895,210.259,210.615,46241,False,False,BULLISH,BULLISH
2026-03-06 12:00:00,210.612,210.882,210.278,210.338,88404,False,False,BULLISH,BULLISH
2026-03-06 16:00:00,210.338,211.397,210.313,211.254,88599,False,False,BULLISH,BULLISH


In [10]:
def swing_point_pos(x):
    if x['is_swing_low']==True:
        return x['Low']-1e-4
    elif x['is_swing_high']==True:
        return x['High']+1e-4
    else:
        return np.nan

df_h4['swing_point'] = df_h4.apply(lambda row: swing_point_pos(row), axis=1)

In [11]:
start_date_chart = '2022-1-1 00:00:00'
end_date_chart = '2022-12-30 00:00:00'
dfpl = df_h4.loc[start_date_chart : end_date_chart]#.set_index("Gmt time")

# Create a plot with 2 rows
fig = make_subplots(rows=1, cols=1)

# Add candlestick plot on the first row
fig.add_trace(go.Candlestick(x=dfpl.index,
                             open=df_h4['Open'],
                             high=df_h4['High'],
                             low=df_h4['Low'],
                             close=df_h4['Close']),
              row=1, col=1)

#Add markers for trade entry points on the same subplot
fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['swing_point'], mode="markers",
                         marker=dict(size=6, color="black"),
                         name="swing point"),
              row=1, col=1)

fig.update_layout(width=1200, height=800, sliders=[])
fig.show()

In [12]:
# supports_list = define_resistant_level(df_h4, define_key_level_window)
supports_list = define_support_level(df_h4, define_key_level_window)
supports_list

[[(Timestamp('2026-02-20 16:00:00'),
   Open                209.391
   High                209.677
   Low                 208.963
   Close               209.193
   Tickvol               81589
   is_swing_high          True
   is_swing_low          False
   market_state        BEARISH
   market_state_d1     BEARISH
   swing_point        209.6771
   Name: 2026-02-20 16:00:00, dtype: object),
  209.63]]

In [13]:
def support_pos(supports_list, candle):
    for support_pt in supports_list:
        if candle.name == support_pt[0][0]:
            # print(candle)
            return support_pt[1]

    return np.nan

def is_support(supports_list, candle):
    for support_pt in supports_list:
        if candle.name == support_pt[0][0]:
            # print(candle)
            return True

    return False

df_h4['support_point'] = df_h4.apply(lambda row: support_pos(supports_list, row), axis=1)
df_h4['is_support'] = df_h4.apply(lambda row: is_support(supports_list, row), axis=1)

In [14]:
start_date_chart = '2025-11-1 00:00:00'
end_date_chart = '2025-12-30 00:00:00'
dfpl = df_h4.loc[start_date_chart : end_date_chart]#.set_index("Gmt time")

# Create a plot with 2 rows
fig = make_subplots(rows=1, cols=1)

# Add candlestick plot on the first row
fig.add_trace(go.Candlestick(x=dfpl.index,
                             open=dfpl['Open'],
                             high=dfpl['High'],
                             low=dfpl['Low'],
                             close=dfpl['Close']),
              row=1, col=1)

#Add markers for trade entry points on the same subplot
fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['support_point'], mode="markers",
                         marker=dict(size=8, color="green"),
                         name="support point"),
              row=1, col=1)

fig.update_layout(width=1500, height=800, sliders=[])
fig.show()

In [15]:
# back candle is the number of day to look back for resistant
# number of day to skip in order to avoid look-ahead
def get_support_list(df, current_index, back_candles_window=20, gap=1):
    start_date = current_index - timedelta(days=back_candles_window)
    end_date = current_index - timedelta(days=gap)

    window = df.loc[start_date : end_date]
    supports_level = window[window['is_support'] == True]
    return supports_level.groupby('support_point')['support_point'].max()

In [16]:
# detect if all candles are above support level
def is_candles_above_support_level_list(df, breakout_candles_above=14, key_level_lookback_days=25):
    valid_support_level_list = []

    for index in range(len(df)):
        candle = df_h4.iloc[index]
        level_list = []
        support_levels_list = get_support_list(df, candle.name, key_level_lookback_days, 1)

        for level in support_levels_list:
            if (df['Low'].iloc[index-breakout_candles_above:index] > level).all():
                level_list.append(level)

        valid_support_level_list.append(level_list)

    return valid_support_level_list

df_h4['valid_support_levels'] = is_candles_above_support_level_list(df_h4, breakout_candles_above, key_level_lookback_days)

In [17]:
# process m15 data
df_m15 = df_m15_raw.loc[start_date - timedelta(hours=3): end_date]
df_m15 = implement_market_structure_states_direct(df_m15, n=structure_m15_window)

m15_daily_direction_list = []
m15_h4_key_level_list = []
m15_is_supports_list = []
m15_is_above_support_list = []

df_m15 = df_m15.loc[start_date : end_date]
for i in range(len(df_h4)):
    row = df_h4.iloc[i]
    start_date_h4 = row.name
    end_date_h4 = start_date_h4 + timedelta(hours=3, minutes=59)
    df_by_day = df_m15.loc[f'{start_date_h4}' : f'{end_date_h4}']

    for index2, row2 in df_by_day.iterrows():
        m15_h4_key_level_list.append(df_h4.iloc[i]['support_point'])
        m15_is_supports_list.append(df_h4.iloc[i]['is_support'])
        m15_is_above_support_list.append(df_h4.iloc[i]['valid_support_levels'])
        # affect current market states of H4 candle because the result is market state of previous D1 candle
        m15_daily_direction_list.append(df_h4.iloc[i]['market_state_d1'])

df_m15['support_point'] = m15_h4_key_level_list
df_m15['is_support'] = m15_is_supports_list
df_m15['valid_support_levels'] = m15_is_above_support_list
df_m15['market_state_d1'] = m15_daily_direction_list

indicator_atr_14 = AverageTrueRange(df_m15['High'], df_m15['Low'], df_m15['Close'], window=14)
df_m15['ATR'] = indicator_atr_14._atr

df_m15

,Open,High,Low,Close,Tickvol,is_swing_high,is_swing_low,market_state,support_point,is_support,valid_support_levels,market_state_d1,ATR
Date,,,,,,,,,,,,,
2026-02-11 00:00:00,210.561,210.607,210.507,210.599,1058,False,False,NaN,NaN,False,[],BULLISH,0.000000
2026-02-11 00:15:00,210.596,210.662,210.583,210.643,1155,True,False,NaN,NaN,False,[],BULLISH,0.000000
2026-02-11 00:30:00,210.646,210.652,210.593,210.608,914,False,False,NaN,NaN,False,[],BULLISH,0.000000
2026-02-11 00:45:00,210.609,210.626,210.599,210.622,579,False,False,NaN,NaN,False,[],BULLISH,0.000000
2026-02-11 01:00:00,210.621,210.643,210.507,210.555,1076,False,False,NaN,NaN,False,[],BULLISH,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-06 22:00:00,211.447,211.507,211.402,211.502,2424,False,False,BULLISH,NaN,False,[],BULLISH,0.155652
2026-03-06 22:15:00,211.498,211.538,211.468,211.471,3441,False,False,BULLISH,NaN,False,[],BULLISH,0.149534
2026-03-06 22:30:00,211.473,211.501,211.338,211.372,3763,False,False,BULLISH,NaN,False,[],BULLISH,0.150496


In [18]:
candle_index = datetime.strptime('2024-02-15 10:15', date_format)
print(get_support_list(df_m15, candle_index, key_level_lookback_days, 1))

Series([], Name: support_point, dtype: float64)


In [19]:
def get_valid_support_level(df, index, breakout_candles_above_lookback=14):
    periode = df.iloc[index-1-breakout_candles_above_lookback*16:index-1]
    valid_support_levels = []

    for candle in periode.iterrows():
        if len(candle[1]['valid_support_levels']) > 0:
            for i in range(len(candle[1]['valid_support_levels'])):
                valid_support_levels.append(candle[1]['valid_support_levels'][i])

    return list(set(valid_support_levels))

# 2024-03-25 06:00:00 2300
print(get_valid_support_level(df_m15, 2961, 30))

[]


In [20]:
# using m15 market structure for entry
def apply_total_signal(df, key_level_lookback_days = 20, gap=1, breakout_candles_above_lookback = 30):
    # Initialize the 'TotalSignal' column
    df_signal_list = []
    start_trade_hour = 5
    stop_trade_hour = 20
    current_day = None
    is_key_level_touched = False
    valid_support_levels = []
    is_above_support_levels = False

    high_tf_condition_long_satisfied = False
    high_tf_condition_long_candle = None
    high_tf_condition_short_satisfied = False
    high_tf_condition_short_candle = None

    # we want market structure to switch from bearish to bullish when htf conditions are met
    # if market structure is already bullish we need to wait
    previous_market_structure = None

    for i in range(0, len(df)):
        current_bar_date = df.index[i]
        signal = 0

        if current_day != current_bar_date.date() or current_bar_date.time().hour >= stop_trade_hour:
            current_day = current_bar_date.date()
            high_tf_condition_long_satisfied = False
            high_tf_condition_short_satisfied = False
            is_key_level_touched = False
            valid_support_levels = []
            is_above_support_levels = False

        # h4 key level conditions
        supports_levels = get_support_list(df, current_bar_date, key_level_lookback_days, gap)
        supports_levels_touched = []
        for level in supports_levels:
            if df['Low'].iloc[i-1] < level:
                # print(level)
                supports_levels_touched.append(level)
                is_key_level_touched = True

        c_time_trade = start_trade_hour < current_bar_date.hour and current_bar_date.hour < stop_trade_hour

        if c_time_trade:
            # d1 trending conditions
            c_long_daily_direction = df['market_state_d1'].iloc[i-1] == 'BULLISH'
            c_m15_ms_bullish = df['market_state'].iloc[i] == 'BULLISH'

            # breakout_candles_above candle needs to be above support level for the breakout to be valid
            valid_support_levels = get_valid_support_level(df, i, breakout_candles_above_lookback)

            for valid_level in valid_support_levels:
                for level_touched in supports_levels_touched:
                    if valid_level == level_touched:
                        # print(f"{current_bar_date} {valid_level} = {level_touched}")
                        # print(f"c_m15_ms_bullish = {c_m15_ms_bullish} & previous_market_structure = {previous_market_structure}")
                        is_above_support_levels = True

            # m15 market structure shift to bullish

            # print(current_bar_date)
            # Combine conditions
            if (c_long_daily_direction and
                is_key_level_touched and
                not high_tf_condition_long_satisfied):
                high_tf_condition_long_satisfied = True
                high_tf_condition_long_candle = current_bar_date
                previous_market_structure = df['market_state'].iloc[i]
                print(f"{current_bar_date} {i}")

            if (high_tf_condition_long_satisfied == True and
                c_m15_ms_bullish and
                high_tf_condition_long_candle < current_bar_date and
                is_above_support_levels and
                previous_market_structure == 'BEARISH'):
                # print(f"high_tf_condition_long_candle = {high_tf_condition_long_candle} & current_bar_date = {current_bar_date}")
                # print(f"supports_levels_touched = {supports_levels_touched}")
                # print(f"valid_support_levels = {valid_support_levels}")
                signal = 2

            # # d1 trending conditions
            # c_short_daily_direction = df['market_state_d1'].iloc[i-1] == 'BEARISH'
            # # c_40_candles_below_ema50 = is_all_candle_below(df[i-ema50_window:i-1], df['EMA_50'].iloc[i-1])

            # # h4 ema conditions
            # c_h4_ema20_below_ema50 = df['EMA_20'].iloc[i-1] < df['EMA_50'].iloc[i-1]
            # c_h4_low_above_ema20 = df['Low'].iloc[i-1] > df['EMA_20'].iloc[i-1]

            # # if after all conditions are met then one candle is above ema50, wait till new setup
            # c_40_candles_below_ema50 = (df['High'].iloc[i-ema50_window:i-1] < df['EMA_50'].iloc[i-1]).all()

            # # m15 market structure shift to bullish
            # c_m15_ms_bearish = df['market_state'].iloc[i-1] == 'BEARISH'
            
            # # Combine conditions
            # if (c_short_daily_direction and
            #     c_h4_ema20_below_ema50 and
            #     c_h4_low_above_ema20 and
            #     high_tf_condition_short_satisfied):
            #     high_tf_condition_short_satisfied = True
            #     high_tf_condition_short_candle = current_bar_date
            #     previous_market_structure = df['market_state'].iloc[i-1]

            # if not c_40_candles_below_ema50:
            #     high_tf_condition_short_satisfied = False
            #     high_tf_condition_short_candle = None
    
            # if (high_tf_condition_short_satisfied == True and
            #     c_m15_ms_bearish and
            #     high_tf_condition_short_candle < current_bar_date and
            #     previous_market_structure == 'BULLISH'):
            #     # print(f"high_tf_condition_long_candle = {high_tf_condition_long_candle} & current_bar_date = {current_bar_date}")
            #     signal = 1

        if previous_market_structure != df['market_state'].iloc[i]:
            previous_market_structure = df['market_state'].iloc[i]    

        df_signal_list.append(signal)

    return df_signal_list

df_m15['TotalSignal'] = apply_total_signal(df=df_m15, key_level_lookback_days=key_level_lookback_days, gap=gap_look_ahead, breakout_candles_above_lookback=breakout_candles_above_lookback)

2026-03-02 06:00:00 1260
2026-03-03 09:15:00 1369
2026-03-04 06:00:00 1452
2026-03-05 06:00:00 1548


In [21]:
# # using m15 market structure for entry improve
# def apply_total_signal(df, key_level_lookback_days = 20, gap=1, breakout_candles_above_lookback = 30):
#     # Initialize the 'TotalSignal' column
#     df_signal_list = []
#     valid_support_levels = []

#     start_trade_hour = 5
#     stop_trade_hour = 20
#     high_tf_condition_validation_duration = 0

#     current_day = None
#     high_tf_condition_long_candle = None
#     high_tf_condition_short_candle = None

#     is_key_level_touched = False
#     is_above_support_levels = False
#     high_tf_condition_long_satisfied = False
#     high_tf_condition_short_satisfied = False
#     is_m15_market_state_valid = False

#     # we want market structure to switch from bearish to bullish when htf conditions are met
#     # if market structure is already bullish we need to wait
#     previous_market_structure = None

#     for i in range(0, len(df)):
#         current_bar_date = df.index[i]
#         signal = 0

#         if current_day != current_bar_date.date() or current_bar_date.time().hour >= stop_trade_hour:
#             current_day = current_bar_date.date()
#             high_tf_condition_short_satisfied = False
#             is_key_level_touched = False
#             valid_support_levels = []
#             is_above_support_levels = False
#             is_m15_market_state_valid = False

#         if high_tf_condition_validation_duration == i:
#             high_tf_condition_long_satisfied = False
#             high_tf_condition_long_candle = None

#         if (high_tf_condition_long_candle is not None and
#             c_m15_ms_bullish and
#             high_tf_condition_long_candle < current_bar_date and
#             previous_market_structure == 'BEARISH'):
#             is_m15_market_state_valid = True

#         # h4 key level conditions
#         supports_levels = get_support_list(df, current_bar_date, key_level_lookback_days, gap)
#         supports_levels_touched = []
#         for level in supports_levels:
#             if df['Low'].iloc[i-1] < level:
#                 # print(level)
#                 supports_levels_touched.append(level)
#                 is_key_level_touched = True

#         c_time_trade = start_trade_hour < current_bar_date.hour and current_bar_date.hour < stop_trade_hour

#         if c_time_trade:
#             # d1 trending conditions
#             c_long_daily_direction = df['market_state_d1'].iloc[i-1] == 'BULLISH'
#             c_m15_ms_bullish = df['market_state'].iloc[i] == 'BULLISH'

#             # breakout_candles_above candle needs to be above support level for the breakout to be valid
#             valid_support_levels = get_valid_support_level(df, i, breakout_candles_above_lookback)

#             for valid_level in valid_support_levels:
#                 for level_touched in supports_levels_touched:
#                     if valid_level == level_touched:
#                         # print(f"{current_bar_date} {valid_level} = {level_touched}")
#                         # print(f"c_m15_ms_bullish = {c_m15_ms_bullish} & previous_market_structure = {previous_market_structure}")
#                         is_above_support_levels = True

#             # m15 market structure shift to bullish

#             # print(current_bar_date)
#             # Combine conditions
#             if (c_long_daily_direction and
#                 is_key_level_touched and
#                 is_above_support_levels and
#                 not high_tf_condition_long_satisfied):
#                 high_tf_condition_long_satisfied = True
#                 high_tf_condition_validation_duration = i + 100
#                 high_tf_condition_long_candle = current_bar_date
#                 previous_market_structure = df['market_state'].iloc[i]
#                 print(f"{current_bar_date} {i}")

#             if (high_tf_condition_long_satisfied == True and
#                 is_m15_market_state_valid):
#                 # print(f"high_tf_condition_long_candle = {high_tf_condition_long_candle} & current_bar_date = {current_bar_date}")
#                 # print(f"supports_levels_touched = {supports_levels_touched}")
#                 # print(f"valid_support_levels = {valid_support_levels}")
#                 signal = 2

#             # # d1 trending conditions
#             # c_short_daily_direction = df['market_state_d1'].iloc[i-1] == 'BEARISH'
#             # # c_40_candles_below_ema50 = is_all_candle_below(df[i-ema50_window:i-1], df['EMA_50'].iloc[i-1])

#             # # h4 ema conditions
#             # c_h4_ema20_below_ema50 = df['EMA_20'].iloc[i-1] < df['EMA_50'].iloc[i-1]
#             # c_h4_low_above_ema20 = df['Low'].iloc[i-1] > df['EMA_20'].iloc[i-1]

#             # # if after all conditions are met then one candle is above ema50, wait till new setup
#             # c_40_candles_below_ema50 = (df['High'].iloc[i-ema50_window:i-1] < df['EMA_50'].iloc[i-1]).all()

#             # # m15 market structure shift to bullish
#             # c_m15_ms_bearish = df['market_state'].iloc[i-1] == 'BEARISH'
            
#             # # Combine conditions
#             # if (c_short_daily_direction and
#             #     c_h4_ema20_below_ema50 and
#             #     c_h4_low_above_ema20 and
#             #     high_tf_condition_short_satisfied):
#             #     high_tf_condition_short_satisfied = True
#             #     high_tf_condition_short_candle = current_bar_date
#             #     previous_market_structure = df['market_state'].iloc[i-1]

#             # if not c_40_candles_below_ema50:
#             #     high_tf_condition_short_satisfied = False
#             #     high_tf_condition_short_candle = None
    
#             # if (high_tf_condition_short_satisfied == True and
#             #     c_m15_ms_bearish and
#             #     high_tf_condition_short_candle < current_bar_date and
#             #     previous_market_structure == 'BULLISH'):
#             #     # print(f"high_tf_condition_long_candle = {high_tf_condition_long_candle} & current_bar_date = {current_bar_date}")
#             #     signal = 1

#         if previous_market_structure != df['market_state'].iloc[i]:
#             previous_market_structure = df['market_state'].iloc[i]    

#         df_signal_list.append(signal)

#     return df_signal_list

# df_m15['TotalSignal'] = apply_total_signal(df=df_m15, key_level_lookback_days=key_level_lookback_days, gap=gap_look_ahead, breakout_candles_above_lookback=breakout_candles_above_lookback)

In [22]:
len(df_m15[df_m15.TotalSignal != 0])

6

In [23]:
def pointpos(x):
    if x['TotalSignal']==2:
        return x['Low']-1e-4
    elif x['TotalSignal']==1:
        return x['High']+1e-4
    else:
        return np.nan

df_m15['pointpos'] = df_m15.apply(lambda row: pointpos(row), axis=1)

In [24]:
# Affichage des transitions d'états m15
df_m15["state_changed"] = df_m15["market_state"] != df_m15[
    "market_state"
].shift(1)
df_state_changed = df_m15[df_m15["state_changed"]][["Close", "market_state"]]

mk_bullish_pos_list = []
mk_bearish_pos_list = []

for i in range(len(df_m15)):
    previous_candle = df_m15.iloc[i-1]
    candle = df_m15.iloc[i]
    if candle['market_state'] == 'BULLISH' and previous_candle['market_state'] != 'BULLISH':
        mk_bullish_pos_list.append(candle['Low'] - 1e-4)
        mk_bearish_pos_list.append(np.nan)
        continue        

    if candle['market_state'] == 'BEARISH' and previous_candle['market_state'] != 'BEARISH':
        mk_bullish_pos_list.append(np.nan)
        mk_bearish_pos_list.append(candle['High'] + 1e-4)
        continue

    mk_bullish_pos_list.append(np.nan)
    mk_bearish_pos_list.append(np.nan)

df_m15['mk_bullish_pos'] = mk_bullish_pos_list
df_m15['mk_bearish_pos'] = mk_bearish_pos_list

In [25]:
start_date_chart = '2026-3-1 00:00:00'
end_date_chart = '2026-3-30 00:00:00'
dfpl = df_m15.loc[start_date_chart : end_date_chart]#.set_index("Gmt time")

# Create a plot with 2 rows
fig = make_subplots(rows=1, cols=1)

# Add candlestick plot on the first row
fig.add_trace(go.Candlestick(x=dfpl.index,
                             open=dfpl['Open'],
                             high=dfpl['High'],
                             low=dfpl['Low'],
                             close=dfpl['Close']),
              row=1, col=1)

#Add markers for trade entry points on the same subplot
# fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['support_point'], mode="markers",
#                          marker=dict(size=8, color="green"),
#                          name="support point"),
#               row=1, col=1)

# Add markers for entry point m15
fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['pointpos'], mode="markers",
                         marker=dict(size=12, color="MediumPurple"),
                         name="entry"),
              row=1, col=1)

# # Add markers for market structure daily = bullish
# fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['mk_bullish_pos'], mode="markers",
#                          marker=dict(size=8, color="green"),
#                          name="bullish"),
#               row=1, col=1)

# # Add markers for market structure daily = bearish
# fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['mk_bearish_pos'], mode="markers",
#                          marker=dict(size=8, color="red"),
#                          name="bearish"),
#               row=1, col=1)

fig.update_layout(width=1500, height=800, sliders=[])
fig.show()

In [26]:
# dfopt = df_m15.loc['2022-01-1 00:00:00' : '2026-9-16 00:00:00']
dfopt = df_m15


In [27]:
# add some sl tp management
def SIGNAL():
    return dfopt.TotalSignal

class MyStrat(Strategy):
    mysize = 0.5
    slcoef = 2
    TPcoef = 3
    risk_percent = 0.01
    last_swing_low = None
    last_swing_high = None
    previous_swing_low = None
    previous_swing_high = None
    previous_market_state = 'BULLISH'
    
    def init(self):
        super().init()
        self.signal1 = self.I(SIGNAL)

    def next(self):
        super().next()

        # if len(self.trades) > 0:
        #     print(self.trades)
        
        # if self.data['is_swing_low'][len(self.data['is_swing_low'])-3] == True:
        #     if self.last_swing_low is None:
        #         self.last_swing_low = self.data.Low[len(self.data['is_swing_low'])-3]
        #     else:
        #         if self.data.Low[len(self.data['is_swing_low'])-3] < self.last_swing_low:
        #             self.last_swing_low = self.data.Low[len(self.data['is_swing_low'])-3]

        # if self.data['is_swing_high'][len(self.data['is_swing_high'])-3] == True:
        #     if self.last_swing_high is None:
        #         self.last_swing_high = self.data.High[len(self.data['is_swing_high'])-3]
        #     else:
        #         if self.last_swing_high > self.data.High[len(self.data['is_swing_high'])-3]:
        #             self.last_swing_high = self.data.High[len(self.data['is_swing_high'])-3]

        if len(self.trades)==0:
            if self.signal1==2:
                sl = abs(self.data.Close[-1] - self.data.Low[len(self.data.Low) - 100 : len(self.data.Low)].min())
                tp = self.TPcoef * sl

                position_size = int((self.risk_percent * self.equity) / sl)

                sl1 = self.data.Close[-1] - sl
                tp1 = self.data.Close[-1] + tp
                # print(f"go long date= {self.data.index[-1]} & sl1 = {sl1} & tp1 = {tp1}")

                # print(f"{self.data.index[-1]} sl = {sl1} & previous_swing_low = {self.previous_swing_low}")
                self.buy(sl=sl1, tp=tp1, size=position_size)

            if self.signal1==1:
                sl = abs(self.data.Close[-1] - self.previous_swing_high)
                # sl = self.data['ATR'][-1] * 1.5

                tp = self.TPcoef * sl
                c_sl_tp = sl < self.data['ATR'][-1] * 2.5 and sl > self.data['ATR'][-1] * 0.5
                position_size = int((self.risk_percent * self.equity) / sl)
                # if c_sl_tp:
                sl1 = self.data.Close[-1] + sl
                tp1 = self.data.Close[-1] - tp
                # print(f"go short date= {self.data.index[-1]} & sl1 = {sl1} & tp1 = {tp1}")
                self.sell(sl=sl1, tp=tp1, size=position_size)

        if self.data['market_state'][-1] != self.previous_market_state:
            if self.last_swing_low is not None and self.last_swing_high is not None:
                self.previous_swing_low = self.last_swing_low
                self.previous_swing_high = self.last_swing_high
            self.last_swing_high = None
            self.last_swing_low = None
            self.previous_market_state = self.data['market_state'][-1] 
            # print(f"previous_swing_low = {self.previous_swing_low} & previous_swing_high = {self.previous_swing_high}")

In [28]:
# add some sl tp management
def SIGNAL():
    return dfopt.TotalSignal

class MyStrat(Strategy):
    mysize = 0.5
    slcoef = 2
    TPcoef = 2
    risk_percent = 0.01
    last_swing_low = None
    last_swing_high = None
    previous_swing_low = None
    previous_swing_high = None
    previous_market_state = 'BULLISH'
    breakout_candles_above_lookback = 0
    key_level_gap = 0.2
    
    def init(self):
        self.current_day = None
        self.order_placed_time = None

        super().init()
        self.breakout_candles_above_lookback = breakout_candles_above_lookback
        self.signal1 = self.I(SIGNAL)


    def reset_range(self, day):
        # delete placed order at the end of the day if not executed
        if self.order_placed_time:
            for order in self.orders:
                order_executed= False
                for trade in self.trades:
                    if trade.tag == order.tag:
                        order_executed = True

                if order_executed == False:
                    order.cancel()

        self.current_day = day
        self.order_placed_time = None

    def get_tag(self):
        return f"{self.order_placed_time}"
    
    def next(self):
        super().next()

        current_bar_date = self.data.index[-1].date()
        if self.current_day != current_bar_date:
            self.reset_range(current_bar_date)

        if len(self.trades)==0:
            if self.signal1[-1]==2:
                valid_support_list = get_valid_support_level(dfopt, len(self.data.index), breakout_candles_above_lookback)
                if len(valid_support_list) > 0:
                    planned_entry_price = 0
                    sl = 0
                    is_entry_in_valid_zone = self.data.Close[-1] > min(valid_support_list) - self.key_level_gap
                    
                    if is_entry_in_valid_zone: 
                        planned_entry_price = self.data.Close[-1]
                        sl = abs(planned_entry_price - self.data.Low[len(self.data.Low) - 100 : len(self.data.Low)].min())
                    else :
                        planned_entry_price = min(valid_support_list) - self.key_level_gap
                        sl = abs((min(valid_support_list) - self.key_level_gap) - self.data.Low[len(self.data.Low) - 100 : len(self.data.Low)].min())
                    
                    tp = self.TPcoef * sl
    
                    position_size = int((self.risk_percent * self.equity) / sl)
    
                    sl1 = planned_entry_price - sl
                    tp1 = planned_entry_price + tp
                    # print(f"go long date= {self.data.index[-1]} & sl1 = {sl1} & tp1 = {tp1}")

                    if is_entry_in_valid_zone: 
                        self.buy(sl=sl1, tp=tp1, size=position_size)
                        print(f"{self.data.index[-1]} sl = {sl1} & sgnal = {self.signal1[-1]}")
                    else:
                        self.buy(sl=sl1, tp=tp1, size=position_size, stop=planned_entry_price)
                        print(f"{self.data.index[-1]} sl = {sl1} & sgnal = {self.signal1[-1]} & planned_entry_price = {planned_entry_price}")

                    self.order_placed_time = self.data.index[-1]
                

In [29]:
bt = Backtest(dfopt, MyStrat, cash=10000, margin=0.02) #0.0002
stats = bt.run()

print(stats)

bt.plot()

2026-03-02 13:00:00 sl = 209.345 & sgnal = 2
2026-03-03 18:30:00 sl = 209.213 & sgnal = 2
2026-03-05 08:30:00 sl = 209.176 & sgnal = 2
Start                     2026-02-11 00:00:00
End                       2026-03-09 00:00:00
Duration                     26 days 00:00:00
Exposure Time [%]                    20.19848
Equity Final [$]                     9996.723
Equity Peak [$]                      10058.46
Return [%]                           -0.03277
Buy & Hold Return [%]                 0.08215
Return (Ann.) [%]                     -0.4578
Volatility (Ann.) [%]                 6.55227
CAGR [%]                             -0.45937
Sharpe Ratio                         -0.06987
Sortino Ratio                        -0.09794
Calmar Ratio                         -0.14806
Alpha [%]                             -0.0641
Beta                                  0.38138
Max. Drawdown [%]                    -3.09209
Avg. Drawdown [%]                    -0.83387
Max. Drawdown Duration        6 days 

GridPlot(id='p1308', ...)